# 09 · Transformer Mathematics

**The big idea:** instead of reading words one at a time, let every token *look at* every other token and decide what matters. That "looking" is a softmax over dot products.

**Where this leads →** an LLM is a big stack of these blocks that predicts the next token (10).

```text
 tokens → embeddings (+ position) → [ Attention → Add&Norm → FFN → Add&Norm ] × N → logits
```

---

## 📂 Notes in this chapter

One short note per concept, grouped into folders. Read in order. Code builds on earlier notes, so `notebook.ipynb` (all code, in order) is the easiest way to run everything.

### `1-inputs/`

- [1. Token Representations](1-inputs/01-token-representations.md)
- [2. Embeddings](1-inputs/02-embeddings.md)
- [3. Positional Encoding](1-inputs/03-positional-encoding.md)

### `2-attention/`

- [4. Query, Key, Value](2-attention/04-query-key-value.md)
- [5. Dot-Product Attention, Scaling & Softmax](2-attention/05-dot-product-attention-scaling-softmax.md)
- [6. Self-Attention](2-attention/06-self-attention.md)

### `3-multi-head-and-ffn/`

- [7. Multi-Head Attention](3-multi-head-and-ffn/07-multi-head-attention.md)
- [8. Feed-Forward Network](3-multi-head-and-ffn/08-feed-forward-network.md)

### `4-stabilizers/`

- [9. Residual Connections](4-stabilizers/09-residual-connections.md)
- [10. Layer Normalization](4-stabilizers/10-layer-normalization.md)


# Token Representations

*📓 Notes · [09 · Transformer Mathematics](../README.md) · inputs*

**📝 In simple words:** Text is chopped into tokens, and each token is an integer ID.

**🔢 Small example:** `"I love AI"` → `["I", "love", "AI"]` → ids `[4, 17, 9]` (toy vocabulary).

**🤖 Why AI uses it:** models can only do math on numbers.

In [ ]:
import numpy as np
vocab = {"I": 4, "love": 17, "AI": 9}
ids = [vocab[t] for t in "I love AI".split()]; print(ids)

**🧪 Try it myself:** Ids for `"AI love I"`? <details><summary>Solution</summary>`[9, 17, 4]`.</details>

---
[Embeddings](../1-inputs/02-embeddings.md) →


# Embeddings

*📓 Notes · [09 · Transformer Mathematics](../README.md) · inputs*

**📝 In simple words:** A lookup table: ID → learned vector.

**💡 Think of it like this:** Similar words end up with nearby vectors (dot product / cosine large).

**📐 Formula to remember:** `E ∈ ℝ^(V×d)`; embedding of token `i` is row `E[i]`.

**✏️ Let me calculate it:** vocab 20, d = 4 → `E` is `20×4`; ids `[4,17,9]` → a `3×4` matrix `X`.

**🤖 Why AI uses it:** gives the network a meaningful continuous space; the embedding table is learned by backprop like any weight.

In [ ]:
rng = np.random.default_rng(0)
E = rng.normal(size=(20, 4))
X = E[ids]; print(X.shape)   # (3, 4)

**🧪 Try it myself:** Vocab 50,000 and d=768: embedding table size? <details><summary>Solution</summary>`50,000 × 768 ≈ 38.4M` numbers.</details>

---
← [Token Representations](../1-inputs/01-token-representations.md)  |  [Positional Encoding](../1-inputs/03-positional-encoding.md) →


# Positional Encoding

*📓 Notes · [09 · Transformer Mathematics](../README.md) · inputs*

**📝 In simple words:** Extra numbers added to say *where* a token is.

**💡 Think of it like this:** Attention itself ignores order ("dog bites man" = "man bites dog" as a set). Position info fixes this.

**📐 Formula to remember (sinusoidal):** `PE(pos, 2i) = sin(pos / 10000^(2i/d))`, `PE(pos, 2i+1) = cos(pos / 10000^(2i/d))`

**✏️ Let me calculate it (d=4, pos=1):** `i=0`: `sin(1)=0.841`, `cos(1)=0.540`; `i=1`: divisor `100` → `sin(0.01)=0.010`, `cos(0.01)=1.000` → `[0.841, 0.540, 0.010, 1.000]`.

**🤖 Why AI uses it:** order matters in language. (Modern LLMs often use rotary/learned positions, same goal.)

In [ ]:
def pos_enc(n, d):
    pe = np.zeros((n, d))
    pos = np.arange(n)[:, None]; i = np.arange(0, d, 2)
    pe[:, 0::2] = np.sin(pos / 10000**(i/d)); pe[:, 1::2] = np.cos(pos / 10000**(i/d))
    return pe
print(pos_enc(3, 4)[1].round(3))   # [0.841 0.54 0.01 1.]
X = X + pos_enc(3, 4)

**🧪 Try it myself:** What is `PE(0, ·)`? <details><summary>Solution</summary>`sin(0)=0`, `cos(0)=1` → `[0,1,0,1]`.</details>

---
← [Embeddings](../1-inputs/02-embeddings.md)  |  [Query, Key, Value](../2-attention/04-query-key-value.md) →


# Query, Key, Value

*📓 Notes · [09 · Transformer Mathematics](../README.md) · attention*

**📝 In simple words:** Three different "views" of each token: `Q = XW_Q`, `K = XW_K`, `V = XW_V`.

**Intuition (library analogy):** Query = what I'm looking for. Key = what each book's label says. Value = the content I read once matched.

**📐 Formula to remember:** `Q = X W_Q`, `K = X W_K`, `V = X W_V`, with `W ∈ ℝ^(d×d_k)` learned.

**🤖 Why AI uses it:** separating "what to match on" from "what to pass along" makes attention flexible.

In [ ]:
d = 4; rng = np.random.default_rng(1)
Wq, Wk, Wv = (rng.normal(size=(d, d)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
print(Q.shape, K.shape, V.shape)   # (3,4) each

**🧪 Try it myself:** Shape of `Q` for 10 tokens, `d=512`, `d_k=64`? <details><summary>Solution</summary>`(10, 64)`.</details>

---
← [Positional Encoding](../1-inputs/03-positional-encoding.md)  |  [Dot-Product Attention, Scaling & Softmax](../2-attention/05-dot-product-attention-scaling-softmax.md) →


# Dot-Product Attention, Scaling & Softmax

*📓 Notes · [09 · Transformer Mathematics](../README.md) · attention*

**📝 In simple words:** Score every token pair with a dot product, softmax into weights, then average the values.

**📐 Formula to remember:** `Attention(Q,K,V) = softmax(QKᵀ / √d_k) V`

**🔢 Small example (2 tokens, d=2):** `Q = K = I`, `V = [[1,2],[3,4]]`.

**✏️ Let me calculate it:**

```text
QKᵀ = [[1,0],[0,1]]  → divide by √2 → [[0.707, 0],[0, 0.707]]
softmax row 1: e^0.707=2.028, e^0=1 → [0.670, 0.330]
output row 1 = 0.670·[1,2] + 0.330·[3,4] = [1.660, 2.660]
output row 2 = [2.340, 3.340]
```

**Why scale by √d_k?** If entries of `q` and `k` have variance 1, `q·k` has variance `d_k`. Big scores make softmax nearly one-hot with tiny gradients, so dividing by `√d_k` keeps variance ≈ 1.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)
def attention(Q, K, V):
    return softmax(Q @ K.T / np.sqrt(K.shape[-1])) @ V
I = np.eye(2); V2 = np.array([[1., 2.], [3., 4.]])
print(attention(I, I, V2).round(3))   # [[1.66 2.66] [2.34 3.34]]

**🧪 Try it myself:** If all scores are equal, what are the attention weights over 4 tokens? <details><summary>Solution</summary>`[0.25, 0.25, 0.25, 0.25]`: a plain average.</details>

---
← [Query, Key, Value](../2-attention/04-query-key-value.md)  |  [Self-Attention](../2-attention/06-self-attention.md) →


# Self-Attention

*📓 Notes · [09 · Transformer Mathematics](../README.md) · attention*

**📝 In simple words:** Attention where `Q`, `K`, `V` all come from the *same* sequence.

**💡 Think of it like this:** In "The animal didn't cross because **it** was tired", the token "it" learns to attend to "animal".

**Causal mask (GPT-style):** a token can't peek at future tokens: set scores above the diagonal to `−∞` before softmax, so those weights become 0.

In [ ]:
def causal_attention(Q, K, V):
    s = Q @ K.T / np.sqrt(K.shape[-1])
    s = np.where(np.triu(np.ones_like(s), 1) == 1, -1e9, s)   # hide the future
    return softmax(s) @ V
print(causal_attention(Q, K, V).shape)   # (3, 4)

**🧪 Try it myself:** In a causal model with 3 tokens, how many attention weights are non-zero in total? <details><summary>Solution</summary>`1 + 2 + 3 = 6` (lower triangle).</details>

---
← [Dot-Product Attention, Scaling & Softmax](../2-attention/05-dot-product-attention-scaling-softmax.md)  |  [Multi-Head Attention](../3-multi-head-and-ffn/07-multi-head-attention.md) →


# Multi-Head Attention

*📓 Notes · [09 · Transformer Mathematics](../README.md) · multi head and ffn*

**📝 In simple words:** Run several attentions in parallel on smaller slices, then glue the results together.

**💡 Think of it like this:** One head might track grammar, another coreference, another position.

**📐 Formula to remember:** `head_i = Attention(XW_Qⁱ, XW_Kⁱ, XW_Vⁱ)`; `MHA = Concat(head₁..head_h) W_O`, with `d_k = d_model / h`.

**✏️ Let me calculate it:** `d_model = 4`, `h = 2` → each head works in 2 dims → concat back to 4.

In [ ]:
h, dk = 2, 2
heads = [attention(Q[:, i*dk:(i+1)*dk], K[:, i*dk:(i+1)*dk], V[:, i*dk:(i+1)*dk]) for i in range(h)]
Wo = rng.normal(size=(4, 4))
out = np.concatenate(heads, axis=-1) @ Wo; print(out.shape)   # (3, 4)

**🧪 Try it myself:** `d_model=768`, 12 heads: `d_k`? <details><summary>Solution</summary>64.</details>

---
← [Self-Attention](../2-attention/06-self-attention.md)  |  [Feed-Forward Network](../3-multi-head-and-ffn/08-feed-forward-network.md) →


# Feed-Forward Network

*📓 Notes · [09 · Transformer Mathematics](../README.md) · multi head and ffn*

**📝 In simple words:** The same small 2-layer MLP applied to each token separately.

**📐 Formula to remember:** `FFN(x) = W₂ · act(W₁x + b₁) + b₂`, with hidden size usually `4·d_model`.

**🤖 Why AI uses it:** attention *mixes* information between tokens; the FFN *processes* each token (much of the model's stored knowledge sits here).

In [ ]:
W1 = rng.normal(size=(4, 16)); W2 = rng.normal(size=(16, 4))
ffn = lambda x: np.maximum(0, x @ W1) @ W2
print(ffn(out).shape)   # (3, 4)

**🧪 Try it myself:** Parameters in an FFN with `d=512`, hidden `2048` (ignore biases)? <details><summary>Solution</summary>`2·512·2048 ≈ 2.1M`.</details>

---
← [Multi-Head Attention](../3-multi-head-and-ffn/07-multi-head-attention.md)  |  [Residual Connections](../4-stabilizers/09-residual-connections.md) →


# Residual Connections

*📓 Notes · [09 · Transformer Mathematics](../README.md) · stabilizers*

**📝 In simple words:** Add the input back: `y = x + F(x)`.

**💡 Think of it like this:** The block only has to learn a *correction*. The gradient has a clean shortcut: `∂y/∂x = 1 + ∂F/∂x`, so it never vanishes through the "1".

**🤖 Why AI uses it:** makes 100-layer networks trainable.

In [ ]:
y = X + out        # residual around attention

**🧪 Try it myself:** If `F(x) = 0`, what does the block do? <details><summary>Solution</summary>Passes `x` through unchanged (identity).</details>

---
← [Feed-Forward Network](../3-multi-head-and-ffn/08-feed-forward-network.md)  |  [Layer Normalization](../4-stabilizers/10-layer-normalization.md) →


# Layer Normalization

*📓 Notes · [09 · Transformer Mathematics](../README.md) · stabilizers*

**📝 In simple words:** Standardize each token's vector, then re-scale with learned `γ`, `β`.

**📐 Formula to remember:** `LN(x) = γ · (x − μ)/√(σ² + ε) + β`

**🔢 Small example:** `x = [1, 2, 3]`.

**✏️ Let me calculate it:** `μ = 2`, `σ² = 2/3`, `σ = 0.816` → `[−1.225, 0, 1.225]`.

**🤖 Why AI uses it:** keeps activations in a stable range so training doesn't explode or stall.

In [ ]:
def layer_norm(x, eps=1e-5):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps)
print(layer_norm(np.array([1., 2., 3.])).round(3))   # [-1.225 0. 1.225]

**Putting it together (one block):**

In [ ]:
def block(x):
    x = layer_norm(x + causal_attention(x @ Wq, x @ Wk, x @ Wv))
    return layer_norm(x + ffn(x))
print(block(X).shape)   # (3, 4)

**🧪 Try it myself:** What is the mean of a layer-normed vector? <details><summary>Solution</summary>0 (before `γ`, `β`).</details>

---
← [Residual Connections](../4-stabilizers/09-residual-connections.md)